### Import libraries

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.models import inception_v3, Inception_V3_Weights
import time
import itertools
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
from functions import train_model, validate_model, image_gen_w_aug, build_model, EarlyStopper

sns.set_theme()


NUM_EPOCHS = 15
NUM_CLASSES = 3
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
train_dir = os.path.join("datasets/train/")
test_dir = os.path.join("datasets/test/")
best_global_acc = 0.0
best_params = None

# Set the parameters to test
param_grid = {
    'lr': [0.02, 0.002, 0.0002],
    'weight_decay': [1e-4, 1e-5],
    'batch_size': [32, 64, 128]
}

### Training model and finding best hyperparameters

In [ ]:
# Generate all unique combinations of hyperparameters
keys, values = zip(*param_grid.items())
experiments = [dict(zip(keys, v)) for v in itertools.product(*values)]

# training and validation
weights = Inception_V3_Weights.DEFAULT
criterion = nn.CrossEntropyLoss()
print("Starting training...\n")
start_time = time.time()

for idx, params in enumerate(experiments):
    print(f"\n================ Running Experiment {idx + 1}/{len(experiments)} ================")
    print(f"Parameters: {params}")

    # Re-generate data loaders if batch_size is a hyperparameter
    train_generator, val_generator, test_generator = image_gen_w_aug(train_dir, test_dir, params['batch_size'])

    # Re-initialise model architecture
    pre_trained_model = inception_v3(weights=weights, aux_logits=True)
    for param in pre_trained_model.parameters():
        param.requires_grad = False

    model = build_model(pre_trained_model, NUM_CLASSES)
    model.to(device)

    optimiser = optim.Adam(
        model.parameters(),
        lr=params['lr'],
        weight_decay=params['weight_decay']
    )

    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimiser, mode='min', factor=0.5, patience=3)
    early_stopper = EarlyStopper(patience=5, min_delta=0.02)
    model_history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": []
    }
    best_val_acc = 0.0

    # 3. Inner training loop (Your original code)
    for epoch in range(NUM_EPOCHS):
        train_loss, train_acc = train_model(model, train_generator, criterion, optimiser, device)
        model_history["train_loss"].append(train_loss)
        model_history["train_acc"].append(train_acc)

        val_loss, val_acc = validate_model(model, val_generator, criterion, device)
        model_history["val_loss"].append(val_loss)
        model_history["val_acc"].append(val_acc)

        scheduler.step(val_loss)

        if val_acc > best_val_acc:
            best_val_acc = val_acc

        if early_stopper.early_stop(val_loss):
            print(f"Early stopping triggered at epoch {epoch + 1}")
            break

    print(f"Experiment {idx + 1} finished. Best Val Acc: {best_val_acc:.2f}%")

    # Keep track of the absolute best configuration
    if best_val_acc > best_global_acc:
        best_global_acc = best_val_acc
        best_params = params
        # Save the absolute best model overall
        torch.save(model.state_dict(), "./models/best_model.pth")

print(f"\nBest Accuracy: {best_global_acc:.2f}%")
print(f"Best Hyperparameters: {best_params}")

training_time = time.time() - start_time
print("=" * 70)
print(f"Training completed in {training_time / 60:.2f} minutes")
print("=" * 70)

### Validate model on test set

In [ ]:
# Load best model
model.load_state_dict(torch.load("./models/best_model.pth"))
print("Best model loaded!")

# Validate on test set
test_loss, test_acc = validate_model(model, test_generator, criterion, device)
print("\nTest metrics")
print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.2f}%")

### Plot model stats over time

In [ ]:
# Model stats over time
for stat in model_history:
    print(f"\n{stat}: \n{model_history[stat]}")
    print()

# plot loss and accuracy per epoch
plt.figure(figsize=(4, 3))
plt.title("Loss per epoch")
actual_epochs = len(model_history["train_loss"])
plt.plot(np.arange(1, actual_epochs + 1), model_history["train_loss"], label="Train Loss")
plt.plot(np.arange(1, actual_epochs + 1), model_history["val_loss"], label="Val Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.savefig("./images/loss_per_epoch.png")
plt.show()

plt.figure(figsize=(4, 3))
plt.title("Accuracy per epoch")
actual_epochs = len(model_history["train_acc"])
plt.plot(np.arange(1, actual_epochs + 1), model_history["train_acc"], label="Train Accuracy")
plt.plot(np.arange(1, actual_epochs + 1), model_history["val_acc"], label="Val Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.savefig("./images/acc_per_epoch.png")
plt.show()